In [ ]:
# %% 셀 1: 모델 + 데이터 로드 (DETR + LFT + heatmap→bbox CNN)
import os, json, random, hashlib, colorsys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
FRAME_DIR = "./data/2_frame_files"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_FRAMES = 2000
MAX_ACTIVE_PER_FRAME = 150
MAX_TEXT_LEN = 200
SCALAR_DIM = 8
TIME_DIM = 3
PATCH_SIZE = 8
N_PATCHES_X = GRID_W // PATCH_SIZE
N_PATCHES_Y = GRID_H // PATCH_SIZE
N_PATCHES = N_PATCHES_X * N_PATCHES_Y

D_MODEL = 256
N_HEADS = 8
N_LAYERS_TEMPORAL = 4
N_LAYERS_SPATIAL = 4
N_LAYERS_DECODER = 6
N_LAYERS_INST = 4
D_FF = 512
DROPOUT = 0.0
INTRA_CHUNK = 256
SPATIAL_CHUNK = 128
SLOT_DECODE_CHUNK = 128
CNN_HIDDEN = 64
CNN_CHUNK = 4096

# heatmap 시각화용 threshold (CNN 출력은 thresh-free, heatmap 시각화에만 사용)
BEST_TH_X = 0.5
BEST_TH_Y = 0.5

CKPT_PATH = "./model/8_layout_slot_xy_lft_cnn/best.pt"

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"], "text_len": len(inst["text"]),
            "start": inst["start_sec"], "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel, "video_name": video_name, "file_id": file_id,
        "instances": inst_list, "stt_segments": stt_segments, "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    by_channel.setdefault(s["channel"], []).append(s)

eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])

print(f"✅ eval 영상 (필터링 전): {len(eval_samples):,}개")


# ── 모델 (DETR + LFT + heatmap→bbox CNN) ──
class IntraFrameModule(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.combine = nn.Sequential(
            nn.Linear(d_model * 3 + 1, d_model), nn.GELU(),
            nn.Linear(d_model, d_model))
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tokens, slot_mask):
        N, K, D = tokens.shape
        dtype = tokens.dtype
        d = D
        slot_tokens = tokens * slot_mask.unsqueeze(-1).float()
        slot_pad = ~slot_mask
        count = slot_mask.sum(dim=1, keepdim=True).float().clamp(min=1)
        count_norm = count / MAX_ACTIVE_PER_FRAME
        masked_tokens = tokens.masked_fill(slot_pad.unsqueeze(-1), 0.0)
        mean_pool = masked_tokens.sum(dim=1) / count
        masked_for_max = tokens.masked_fill(slot_pad.unsqueeze(-1), float("-inf"))
        max_pool = masked_for_max.max(dim=1).values
        all_pad = ~slot_mask.any(dim=1, keepdim=True)
        max_pool = max_pool.masked_fill(all_pad, 0.0)
        q = self.pool_query.expand(N, -1, -1)
        a = torch.bmm(q, tokens.transpose(1, 2)) / (d ** 0.5)
        a = a.masked_fill(slot_pad.unsqueeze(1), float("-inf"))
        a = F.softmax(a, dim=-1)
        a = a.masked_fill(a.isnan(), 0.0)
        attn_pool = torch.bmm(a, tokens).squeeze(1)
        combined = torch.cat([attn_pool, mean_pool, max_pool, count_norm], dim=-1)
        pooled = self.norm(self.combine(combined))
        pooled = pooled.masked_fill(all_pad, 0.0)
        return pooled.to(dtype), slot_tokens.to(dtype)


class SlotXYDecoderLayer(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout
        self.sa_qkv = nn.Linear(d_model, d_model * 3)
        self.sa_out = nn.Linear(d_model, d_model)
        self.self_norm = nn.LayerNorm(d_model)
        self.self_ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.self_ffn_norm = nn.LayerNorm(d_model)
        self.ca_q = nn.Linear(d_model, d_model)
        self.ca_kv = nn.Linear(d_model, d_model * 2)
        self.ca_out = nn.Linear(d_model, d_model)
        self.cross_norm = nn.LayerNorm(d_model)
        self.cross_ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.cross_ffn_norm = nn.LayerNorm(d_model)

    def _reshape(self, x):
        B, S, _ = x.shape
        return x.reshape(B, S, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, q, patch_features, slot_pad):
        B, S, D = q.shape
        valid = (~slot_pad).unsqueeze(-1).float()
        q_masked = q * valid
        qkv = self.sa_qkv(q_masked)
        sq, sk, sv = qkv.chunk(3, dim=-1)
        sq, sk, sv = self._reshape(sq), self._reshape(sk), self._reshape(sv)
        attn_mask = torch.zeros(B, 1, 1, S, device=q.device, dtype=q.dtype)
        attn_mask = attn_mask.masked_fill(slot_pad[:, None, None, :], float("-inf"))
        sa_out = F.scaled_dot_product_attention(
            sq, sk, sv, attn_mask=attn_mask, dropout_p=0.0)
        sa_out = sa_out.transpose(1, 2).reshape(B, S, D)
        sa_out = self.sa_out(sa_out)
        q = self.self_norm(q_masked + sa_out)
        q = self.self_ffn_norm(q + self.self_ffn(q))
        q = q * valid
        cq = self._reshape(self.ca_q(q))
        ckv = self.ca_kv(patch_features)
        ck, cv = ckv.chunk(2, dim=-1)
        ck, cv = self._reshape(ck), self._reshape(cv)
        ca_out = F.scaled_dot_product_attention(cq, ck, cv)
        ca_out = ca_out.transpose(1, 2).reshape(B, S, D)
        ca_out = self.ca_out(ca_out)
        q = self.cross_norm(q + ca_out)
        q = self.cross_ffn_norm(q + self.cross_ffn(q))
        q = q * valid
        return q


class SlotXYDecoder(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT,
                 n_layers=N_LAYERS_DECODER):
        super().__init__()
        self.n_layers = n_layers
        self.query_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.query_norm = nn.LayerNorm(d_model)
        self.layers = nn.ModuleList([
            SlotXYDecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.x_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_W))
            for _ in range(n_layers)
        ])
        self.y_heads = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, GRID_H))
            for _ in range(n_layers)
        ])
        self.ref_init = nn.Linear(d_model, 2)
        self.ref_update = nn.ModuleList([nn.Linear(d_model, 2) for _ in range(n_layers)])
        self.ref_pos_proj = nn.Sequential(
            nn.Linear(2, d_model), nn.GELU(), nn.Linear(d_model, d_model),
        )

    def forward(self, patch_features, slot_query_input, slot_mask):
        q = self.query_norm(self.query_proj(slot_query_input))
        slot_pad = ~slot_mask
        ref_logit = self.ref_init(q)
        x_aux, y_aux = [], []
        for i, layer in enumerate(self.layers):
            ref = torch.sigmoid(ref_logit)
            ref_emb = self.ref_pos_proj(ref)
            q_cond = q + ref_emb
            q = layer(q_cond, patch_features, slot_pad)
            x_aux.append(self.x_heads[i](q))
            y_aux.append(self.y_heads[i](q))
            delta = self.ref_update[i](q)
            ref_logit = ref_logit + delta
        return x_aux[-1], y_aux[-1], x_aux, y_aux, q, ref_logit


class HeatmapToBBoxCNN(nn.Module):
    def __init__(self, grid=GRID_W, n_aux=N_LAYERS_DECODER, d_model=D_MODEL,
                 hidden=CNN_HIDDEN, min_size=1.0):
        super().__init__()
        self.grid = grid
        self.min_size = min_size
        in_ch = n_aux + 1
        self.encoder = nn.Sequential(
            nn.Conv1d(in_ch, hidden, 5, padding=2), nn.GELU(),
            nn.Conv1d(hidden, hidden, 5, padding=2), nn.GELU(),
            nn.Conv1d(hidden, hidden, 3, padding=1, stride=2), nn.GELU(),
            nn.Conv1d(hidden, hidden, 3, padding=1, stride=2), nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.slot_proj = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(),
            nn.Linear(hidden, hidden),
        )
        self.head = nn.Linear(hidden * 2, 2)

    def forward(self, aux_probs, slot_emb, ref_axis):
        N = aux_probs.shape[0]
        ref = ref_axis.reshape(N, 1, 1).expand(-1, -1, self.grid)
        x = torch.cat([aux_probs, ref], dim=1)
        seq_feat = self.encoder(x).squeeze(-1)
        slot_feat = self.slot_proj(slot_emb)
        combined = torch.cat([seq_feat, slot_feat], dim=-1)
        out = self.head(combined)
        center = torch.sigmoid(out[:, 0]) * self.grid
        size   = torch.sigmoid(out[:, 1]) * (self.grid - self.min_size) + self.min_size
        return center, size


class ViTPatchMaskModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers_temporal=N_LAYERS_TEMPORAL, n_layers_spatial=N_LAYERS_SPATIAL,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.ch_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.vid_proj = nn.Sequential(nn.Linear(emb_dim + 1, d_model // 2), nn.GELU())
        self.stt_proj = nn.Sequential(nn.Linear(emb_dim + 2, d_model // 2), nn.GELU())
        self.len_proj = nn.Sequential(nn.Linear(4, d_model // 2), nn.GELU())
        self.slot_combine = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, d_model))

        inst_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.inst_self_attn = nn.TransformerEncoder(
            inst_layer, num_layers=N_LAYERS_INST, enable_nested_tensor=False)

        self.slot_pos_emb = nn.Embedding(MAX_ACTIVE_PER_FRAME, d_model)
        self.intra_frame = IntraFrameModule(d_model)

        self.channel_emb = nn.Embedding(n_channels, d_model)
        self.time_proj = nn.Sequential(
            nn.Linear(TIME_DIM, 64), nn.GELU(), nn.Linear(64, d_model))
        self.temporal_norm = nn.LayerNorm(d_model)
        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.temporal_transformer = nn.TransformerEncoder(
            temporal_layer, num_layers=n_layers_temporal, enable_nested_tensor=False)

        self.patch_pos_emb = nn.Parameter(torch.randn(1, N_PATCHES, d_model) * 0.02)
        self.spatial_norm = nn.LayerNorm(d_model)
        spatial_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu")
        self.spatial_transformer = nn.TransformerEncoder(
            spatial_layer, num_layers=n_layers_spatial, enable_nested_tensor=False)

        self.slot_decoder = SlotXYDecoder(d_model, n_heads, d_ff, dropout)
        self.x_bbox_cnn = HeatmapToBBoxCNN(grid=GRID_W)
        self.y_bbox_cnn = HeatmapToBBoxCNN(grid=GRID_H)

    def forward(self, channel_ids, telop_mask, time_feats, frame_mask,
                slot_inst_idx, inst_diff_ch, inst_diff_vid, inst_diff_stt, inst_scalars):
        B, T, K = telop_mask.shape
        device = telop_mask.device
        dtype = inst_diff_ch.dtype
        BT = B * T
        D = D_MODEL
        n_aux = self.slot_decoder.n_layers

        ch_input = torch.cat([inst_diff_ch, inst_scalars[..., 1:2]], dim=-1)
        vid_input = torch.cat([inst_diff_vid, inst_scalars[..., 2:3]], dim=-1)
        stt_input = torch.cat([inst_diff_stt,
                               inst_scalars[..., 3:4],
                               inst_scalars[..., 4:5]], dim=-1)
        len_input = torch.cat([inst_scalars[..., 0:1], inst_scalars[..., 5:8]], dim=-1)

        proj_ch  = self.ch_proj(ch_input)
        proj_vid = self.vid_proj(vid_input)
        proj_stt = self.stt_proj(stt_input)
        proj_len = self.len_proj(len_input)
        inst_tokens = self.slot_combine(
            torch.cat([proj_ch, proj_vid, proj_stt, proj_len], dim=-1))

        inst_mask = (inst_scalars[..., 0] != 0)
        inst_pad = ~inst_mask
        inst_tokens = self.inst_self_attn(inst_tokens, src_key_padding_mask=inst_pad)

        smask_flat = telop_mask.reshape(BT, K)
        idx_flat = slot_inst_idx.reshape(BT, K).clamp(min=0)
        batch_for_bt = torch.arange(B, device=device).unsqueeze(1).expand(-1, T).reshape(BT)

        slot_idx = torch.arange(K, device=device)
        pos_emb = self.slot_pos_emb(slot_idx).unsqueeze(0)

        pooled_list, slot_tok_list = [], []
        for start in range(0, BT, INTRA_CHUNK):
            end = min(start + INTRA_CHUNK, BT)
            chunk_bidx = batch_for_bt[start:end].unsqueeze(1).expand(-1, K)
            chunk_iidx = idx_flat[start:end]
            chunk_tokens = inst_tokens[chunk_bidx, chunk_iidx] + pos_emb
            p, s = self.intra_frame(chunk_tokens, smask_flat[start:end])
            pooled_list.append(p)
            slot_tok_list.append(s)

        frame_tokens = torch.cat(pooled_list, dim=0).reshape(B, T, D)
        all_slot_tokens = torch.cat(slot_tok_list, dim=0).reshape(B, T, K, D)

        ch = self.channel_emb(channel_ids).unsqueeze(1).expand(-1, T, -1)
        time = self.time_proj(time_feats)
        tokens = self.temporal_norm(frame_tokens + ch + time)
        temporal_out = self.temporal_transformer(tokens, src_key_padding_mask=~frame_mask)

        temporal_flat = temporal_out.reshape(BT, D)
        slot_tok_flat = all_slot_tokens.reshape(BT, K, D)
        valid_flat = frame_mask.reshape(BT) & smask_flat.any(dim=1)
        valid_idx = valid_flat.nonzero(as_tuple=True)[0]
        n_valid = valid_idx.shape[0]

        all_x = torch.zeros(BT, K, GRID_W, device=device, dtype=dtype)
        all_y = torch.zeros(BT, K, GRID_H, device=device, dtype=dtype)
        aux_x_layers = [torch.zeros(BT, K, GRID_W, device=device, dtype=dtype) for _ in range(n_aux)]
        aux_y_layers = [torch.zeros(BT, K, GRID_H, device=device, dtype=dtype) for _ in range(n_aux)]
        all_cx = torch.zeros(BT, K, device=device, dtype=dtype)
        all_cy = torch.zeros(BT, K, device=device, dtype=dtype)
        all_w  = torch.zeros(BT, K, device=device, dtype=dtype)
        all_h  = torch.zeros(BT, K, device=device, dtype=dtype)

        if n_valid > 0:
            valid_ctx = temporal_flat[valid_idx]
            valid_slot = slot_tok_flat[valid_idx]
            valid_smask = smask_flat[valid_idx]

            queries = valid_ctx.unsqueeze(1) + self.patch_pos_emb
            queries = self.spatial_norm(queries)
            patch_feat_list = []
            for start in range(0, n_valid, SPATIAL_CHUNK):
                end = min(start + SPATIAL_CHUNK, n_valid)
                spatial_out = self.spatial_transformer(queries[start:end])
                patch_feat_list.append(spatial_out)
            patch_features = torch.cat(patch_feat_list, dim=0)

            ctx_expand = valid_ctx.unsqueeze(1).expand(-1, K, -1)
            slot_query_input = torch.cat([ctx_expand, valid_slot], dim=-1)

            x_main_chunks, y_main_chunks = [], []
            x_aux_chunks = [[] for _ in range(n_aux)]
            y_aux_chunks = [[] for _ in range(n_aux)]
            slot_q_chunks, ref_chunks = [], []

            for start in range(0, n_valid, SLOT_DECODE_CHUNK):
                end = min(start + SLOT_DECODE_CHUNK, n_valid)
                xo, yo, xa, ya, q_final, ref_final = self.slot_decoder(
                    patch_features[start:end],
                    slot_query_input[start:end],
                    valid_smask[start:end],
                )
                x_main_chunks.append(xo); y_main_chunks.append(yo)
                for li in range(n_aux):
                    x_aux_chunks[li].append(xa[li])
                    y_aux_chunks[li].append(ya[li])
                slot_q_chunks.append(q_final)
                ref_chunks.append(ref_final)

            all_x[valid_idx] = torch.cat(x_main_chunks, dim=0).to(dtype)
            all_y[valid_idx] = torch.cat(y_main_chunks, dim=0).to(dtype)
            for li in range(n_aux):
                aux_x_layers[li][valid_idx] = torch.cat(x_aux_chunks[li], dim=0).to(dtype)
                aux_y_layers[li][valid_idx] = torch.cat(y_aux_chunks[li], dim=0).to(dtype)

            slot_q_valid = torch.cat(slot_q_chunks, dim=0)
            ref_valid    = torch.cat(ref_chunks, dim=0)
            ref_sig = torch.sigmoid(ref_valid)

            n_total = n_valid * K
            active_flat_mask = valid_smask.reshape(-1)
            active_idx_flat  = active_flat_mask.nonzero(as_tuple=True)[0]
            n_active = active_idx_flat.shape[0]

            cx_flat_full = torch.zeros(n_total, device=device, dtype=dtype)
            cy_flat_full = torch.zeros(n_total, device=device, dtype=dtype)
            w_flat_full  = torch.zeros(n_total, device=device, dtype=dtype)
            h_flat_full  = torch.zeros(n_total, device=device, dtype=dtype)

            if n_active > 0:
                aux_x_stack = torch.stack(
                    [torch.cat(x_aux_chunks[li], dim=0) for li in range(n_aux)], dim=2)
                aux_y_stack = torch.stack(
                    [torch.cat(y_aux_chunks[li], dim=0) for li in range(n_aux)], dim=2)

                aux_x_prob = torch.sigmoid(aux_x_stack.float()).reshape(n_total, n_aux, GRID_W)
                aux_y_prob = torch.sigmoid(aux_y_stack.float()).reshape(n_total, n_aux, GRID_H)
                slot_q_flat = slot_q_valid.reshape(n_total, D).float()
                ref_x_flat  = ref_sig[..., 0].reshape(n_total).float()
                ref_y_flat  = ref_sig[..., 1].reshape(n_total).float()

                aux_x_act  = aux_x_prob[active_idx_flat]
                aux_y_act  = aux_y_prob[active_idx_flat]
                slot_q_act = slot_q_flat[active_idx_flat]
                ref_x_act  = ref_x_flat[active_idx_flat]
                ref_y_act  = ref_y_flat[active_idx_flat]

                cx_chunks_a, w_chunks_a = [], []
                cy_chunks_a, h_chunks_a = [], []
                for s in range(0, n_active, CNN_CHUNK):
                    e = min(s + CNN_CHUNK, n_active)
                    cx_c, w_c = self.x_bbox_cnn(aux_x_act[s:e], slot_q_act[s:e], ref_x_act[s:e])
                    cy_c, h_c = self.y_bbox_cnn(aux_y_act[s:e], slot_q_act[s:e], ref_y_act[s:e])
                    cx_chunks_a.append(cx_c); w_chunks_a.append(w_c)
                    cy_chunks_a.append(cy_c); h_chunks_a.append(h_c)

                cx_flat_full[active_idx_flat] = torch.cat(cx_chunks_a, dim=0).to(dtype)
                cy_flat_full[active_idx_flat] = torch.cat(cy_chunks_a, dim=0).to(dtype)
                w_flat_full[active_idx_flat]  = torch.cat(w_chunks_a, dim=0).to(dtype)
                h_flat_full[active_idx_flat]  = torch.cat(h_chunks_a, dim=0).to(dtype)

            all_cx[valid_idx] = cx_flat_full.reshape(n_valid, K)
            all_cy[valid_idx] = cy_flat_full.reshape(n_valid, K)
            all_w[valid_idx]  = w_flat_full.reshape(n_valid, K)
            all_h[valid_idx]  = h_flat_full.reshape(n_valid, K)

        main_x = all_x.reshape(B, T, K, GRID_W)
        main_y = all_y.reshape(B, T, K, GRID_H)
        aux_x_out = [a.reshape(B, T, K, GRID_W) for a in aux_x_layers]
        aux_y_out = [a.reshape(B, T, K, GRID_H) for a in aux_y_layers]
        params = torch.stack([all_cx, all_cy, all_w, all_h], dim=-1).reshape(B, T, K, 4)
        return main_x, main_y, aux_x_out, aux_y_out, params


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

if "channel2id" in ckpt:
    ckpt_ch2id = ckpt["channel2id"]
    if len(ckpt_ch2id) != len(channel2id):
        channel2id = ckpt_ch2id
        channels = sorted(channel2id.keys(), key=lambda c: channel2id[c])
        eval_samples = [s for s in eval_samples if s["channel"] in channel2id]
        print(f"✅ 학습 채널 필터링 후 eval: {len(eval_samples):,}개")

model = ViTPatchMaskModel(n_channels=len(channels)).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"✅ 모델 로드: {CKPT_PATH}")
print(f"   eval_loss={ckpt['eval_loss']:.4f}")
em = ckpt.get('eval_metrics', {})
if em:
    em_str = "  ".join(f"{k}={v:.3f}" for k, v in em.items() if isinstance(v, (int, float)))
    print(f"   eval_metrics: {em_str}")


In [ ]:
# %% 셀 2: 헬퍼 함수 — CNN 출력을 bbox로 직접 사용 (threshold-free)
import cv2
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

plt.rc('font', family='NanumGothic')

OUTPUT_DIR = "./data/8_layout_DETR-LFT-CNN_test"
VIS_FPS = 10
VIS_W = 288
VIS_H = 512
GAP = 6
HEADER_H = 50
N_PANELS = 5
total_w = VIS_W * N_PANELS + GAP * (N_PANELS - 1)
total_h = VIS_H + HEADER_H
PANEL_NAMES = ["Original", "GT BBox", "Merged Heatmap", "Slot Heatmap", "Pred BBox (CNN)"]

sx = VIS_W / GRID_W
sy = VIS_H / GRID_H

cmap_merged = matplotlib.colormaps["coolwarm"]

try:
    PIL_FONT = ImageFont.truetype("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", 20)
    PIL_FONT_SM = ImageFont.truetype("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", 16)
except:
    PIL_FONT = ImageFont.load_default()
    PIL_FONT_SM = ImageFont.load_default()


def get_instance_colors(n_inst):
    colors = []
    for i in range(n_inst):
        hue = (i * 0.618033988749895) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 0.95)
        colors.append((int(r * 255), int(g * 255), int(b * 255)))
    return colors


def get_swap_channel(orig_channel, video_name, candidates):
    h = hashlib.sha256(f"{orig_channel}||{video_name}".encode()).hexdigest()
    seed = int(h[:8], 16)
    r = random.Random(seed)
    pool = [c for c in candidates if c != orig_channel]
    return r.choice(pool)


def prepare_inputs(sample):
    instances = sample["instances"]
    duration = max(sample["duration"], 0.1)
    n_frames = max(1, min(int(duration * FPS) + 1, MAX_FRAMES))
    n_inst = len(instances)

    times = np.arange(n_frames, dtype=np.float32) / FPS
    time_feats = np.zeros((n_frames, TIME_DIM), dtype=np.float32)
    t_norm = times / max(duration, 1.0)
    time_feats[:, 0] = t_norm
    time_feats[:, 1] = np.sin(2 * np.pi * t_norm)
    time_feats[:, 2] = np.cos(2 * np.pi * t_norm)

    if n_inst == 0:
        return {
            "telop_mask": np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_),
            "gt_masks": np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32),
            "active_matrix": np.zeros((n_frames, 0), dtype=np.bool_),
            "slot_inst_map": np.full((n_frames, MAX_ACTIVE_PER_FRAME), -1, dtype=np.int64),
            "time_feats": time_feats,
            "inst_diff_ch": torch.zeros(1, EMB_DIM),
            "inst_diff_vid": torch.zeros(1, EMB_DIM),
            "inst_diff_stt": torch.zeros(1, EMB_DIM),
            "inst_scalars": np.zeros((1, SCALAR_DIM), dtype=np.float32),
            "n_frames": n_frames, "n_inst": 0,
        }

    inst_starts = np.array([inst["start"] for inst in instances])
    inst_ends   = np.array([inst["end"]   for inst in instances])
    inst_tlens  = np.array([inst["text_len"] for inst in instances])
    inst_cx = np.array([inst["x"] for inst in instances])
    inst_cy = np.array([inst["y"] for inst in instances])
    inst_w  = np.array([inst["w"] for inst in instances])
    inst_h  = np.array([inst["h"] for inst in instances])

    inst_x0 = np.maximum(0, inst_cx - inst_w // 2)
    inst_y0 = np.maximum(0, inst_cy - inst_h // 2)
    inst_x1 = np.minimum(GRID_W, inst_x0 + inst_w)
    inst_y1 = np.minimum(GRID_H, inst_y0 + inst_h)

    channel_emb = F.normalize(text2emb.get(sample["channel"], ZERO_EMB), dim=-1)
    video_emb   = F.normalize(text2emb.get(sample["video_name"], ZERO_EMB), dim=-1)
    inst_embs   = torch.stack([text2emb.get(inst["text"], ZERO_EMB) for inst in instances])
    inst_embs   = F.normalize(inst_embs, dim=-1)

    inst_diff_ch  = inst_embs - channel_emb.unsqueeze(0)
    inst_diff_vid = inst_embs - video_emb.unsqueeze(0)
    inst_diff_stt = torch.zeros(n_inst, EMB_DIM)

    ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
    vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()

    stt_sims = np.zeros(n_inst, dtype=np.float32)
    has_stts = np.zeros(n_inst, dtype=np.float32)
    stt_segments = sample["stt_segments"]
    if len(stt_segments) > 0:
        stt_starts = np.array([seg["start"] for seg in stt_segments])
        stt_ends   = np.array([seg["end"]   for seg in stt_segments])
        stt_embs_raw = torch.stack([text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments])
        stt_embs = F.normalize(stt_embs_raw, dim=-1)
        for i in range(n_inst):
            mid = (inst_starts[i] + inst_ends[i]) / 2
            stt_active = (stt_starts <= mid) & (stt_ends > mid)
            stt_active_idx = np.where(stt_active)[0]
            if len(stt_active_idx) > 0:
                inst_diff_stt[i] = inst_embs[i] - stt_embs[stt_active_idx[0]]
                stt_sims[i] = F.cosine_similarity(
                    inst_embs[i].unsqueeze(0), stt_embs[stt_active_idx[0]].unsqueeze(0)).item()
                has_stts[i] = 1.0

    inst_scalars = np.zeros((n_inst, SCALAR_DIM), dtype=np.float32)
    inst_scalars[:, 0] = inst_tlens / MAX_TEXT_LEN
    inst_scalars[:, 1] = ch_sims
    inst_scalars[:, 2] = vid_sims
    inst_scalars[:, 3] = stt_sims
    inst_scalars[:, 4] = has_stts
    inst_scalars[:, 5] = inst_starts / max(duration, 0.1)
    inst_scalars[:, 6] = inst_ends / max(duration, 0.1)
    inst_scalars[:, 7] = (inst_ends - inst_starts) / max(duration, 0.1)

    active_matrix = (inst_starts[None, :] <= times[:, None] + 0.05) & \
                    (inst_ends[None, :] > times[:, None])

    telop_mask    = np.zeros((n_frames, MAX_ACTIVE_PER_FRAME), dtype=np.bool_)
    slot_inst_map = np.full((n_frames, MAX_ACTIVE_PER_FRAME), -1, dtype=np.int64)
    for fi in range(n_frames):
        active_idx = np.where(active_matrix[fi])[0]
        if len(active_idx) > 0:
            sorted_order = np.argsort(inst_tlens[active_idx])[::-1][:MAX_ACTIVE_PER_FRAME]
            sorted_idx = active_idx[sorted_order]
            for sj, ai in enumerate(sorted_idx):
                telop_mask[fi, sj] = True
                slot_inst_map[fi, sj] = ai

    gt_masks = np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)
    for j in range(n_inst):
        x0, y0, x1, y1 = int(inst_x0[j]), int(inst_y0[j]), int(inst_x1[j]), int(inst_y1[j])
        if x1 <= x0 or y1 <= y0:
            continue
        active_frames = np.where(active_matrix[:, j])[0]
        if len(active_frames) > 0:
            gt_masks[active_frames[:, None, None],
                     np.arange(y0, y1)[None, :, None],
                     np.arange(x0, x1)[None, None, :]] = 1.0

    return {
        "telop_mask": telop_mask,
        "gt_masks": gt_masks,
        "active_matrix": active_matrix,
        "slot_inst_map": slot_inst_map,
        "time_feats": time_feats,
        "inst_diff_ch": inst_diff_ch,
        "inst_diff_vid": inst_diff_vid,
        "inst_diff_stt": inst_diff_stt,
        "inst_scalars": inst_scalars,
        "n_frames": n_frames, "n_inst": n_inst,
    }


def run_inference(inputs, cond_channel_id):
    """CNN params를 직접 bbox로 사용. heatmap도 시각화용 반환."""
    telop_mask = inputs["telop_mask"]
    n_frames = inputs["n_frames"]
    n_inst = inputs["n_inst"]

    if n_inst == 0:
        return {
            "merged_heatmap": np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32),
            "slot_x": np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, GRID_W), dtype=np.float32),
            "slot_y": np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, GRID_H), dtype=np.float32),
            "params": np.zeros((n_frames, MAX_ACTIVE_PER_FRAME, 4), dtype=np.float32),
            "pred_merged_bbox": np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32),
        }

    with torch.no_grad():
        cid = torch.tensor([cond_channel_id], dtype=torch.long, device=device)
        tm_t  = torch.from_numpy(telop_mask).unsqueeze(0).to(device)
        ti_t  = torch.from_numpy(inputs["time_feats"]).unsqueeze(0).to(device)
        fm_t  = torch.ones(1, n_frames, dtype=torch.bool, device=device)
        sim_t = torch.from_numpy(inputs["slot_inst_map"]).unsqueeze(0).to(device)
        dc_t  = inputs["inst_diff_ch"].unsqueeze(0).to(device)
        dv_t  = inputs["inst_diff_vid"].unsqueeze(0).to(device)
        ds_t  = inputs["inst_diff_stt"].unsqueeze(0).to(device)
        sc_t  = torch.from_numpy(inputs["inst_scalars"]).unsqueeze(0).to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            main_x, main_y, aux_x, aux_y, params = model(
                cid, tm_t, ti_t, fm_t, sim_t, dc_t, dv_t, ds_t, sc_t,
            )

        x_prob = torch.sigmoid(main_x[0].float()).cpu().numpy()
        y_prob = torch.sigmoid(main_y[0].float()).cpu().numpy()
        params_np = params[0].float().cpu().numpy()  # [T, K, 4]

    # heatmap 기반 merged (시각화 비교용, threshold)
    merged_hm = np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)
    for fi in range(n_frames):
        active = np.where(telop_mask[fi])[0]
        if len(active) > 0:
            x_bin = x_prob[fi, active] > BEST_TH_X
            y_bin = y_prob[fi, active] > BEST_TH_Y
            slot_2d = y_bin[:, :, None].astype(np.float32) * x_bin[:, None, :].astype(np.float32)
            merged_hm[fi] = slot_2d.max(axis=0)

    # CNN bbox 기반 merged (threshold-free, 메인 메트릭)
    merged_bbox = np.zeros((n_frames, GRID_H, GRID_W), dtype=np.float32)
    for fi in range(n_frames):
        active = np.where(telop_mask[fi])[0]
        for si in active:
            cx, cy, w, h = params_np[fi, si]
            x0 = int(np.clip(round(cx - w / 2), 0, GRID_W))
            y0 = int(np.clip(round(cy - h / 2), 0, GRID_H))
            x1 = int(np.clip(round(cx + w / 2), 0, GRID_W))
            y1 = int(np.clip(round(cy + h / 2), 0, GRID_H))
            if x1 > x0 and y1 > y0:
                merged_bbox[fi, y0:y1, x0:x1] = 1.0

    return {
        "merged_heatmap": merged_hm,
        "pred_merged_bbox": merged_bbox,
        "slot_x": x_prob,
        "slot_y": y_prob,
        "params": params_np,
    }


def compute_eval_metrics(gt_masks, pred_merged, active_matrix):
    gt_bin = gt_masks.astype(np.bool_)
    pred_bool = pred_merged.astype(np.bool_)
    tp_all = (pred_bool & gt_bin).sum()
    fp_all = (pred_bool & ~gt_bin).sum()
    fn_all = (~pred_bool & gt_bin).sum()
    p_all = tp_all / (tp_all + fp_all + 1e-8)
    r_all = tp_all / (tp_all + fn_all + 1e-8)
    f1_all = 2 * p_all * r_all / (p_all + r_all + 1e-8)
    intersection = (pred_bool & gt_bin).sum()
    union = (pred_bool | gt_bin).sum()
    iou_all = intersection / (union + 1e-8)
    gt_pos = gt_bin.sum()
    pred_pos = pred_bool.sum()
    coverage_ratio = pred_pos / (gt_pos + 1e-8)

    n_frames = gt_masks.shape[0]
    frame_metrics = []
    for fi in range(n_frames):
        gt_f = gt_bin[fi]; pred_f = pred_bool[fi]
        tp = (pred_f & gt_f).sum(); fp = (pred_f & ~gt_f).sum(); fn = (~pred_f & gt_f).sum()
        p = tp / (tp + fp + 1e-8); r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        inter = (pred_f & gt_f).sum(); uni = (pred_f | gt_f).sum()
        iou = inter / (uni + 1e-8)
        n_active = int(active_matrix[fi].sum()) if active_matrix.shape[1] > 0 else 0
        frame_metrics.append({
            "frame": fi, "t_sec": round(fi / FPS, 1), "n_active": n_active,
            "gt_positive": int(gt_f.sum()), "pred_positive": int(pred_f.sum()),
            "P": round(float(p), 4), "R": round(float(r), 4),
            "F1": round(float(f1), 4), "IoU": round(float(iou), 4),
        })

    return {
        "aggregate": {
            "P": round(float(p_all), 4), "R": round(float(r_all), 4),
            "F1": round(float(f1_all), 4), "IoU": round(float(iou_all), 4),
            "coverage_ratio": round(float(coverage_ratio), 4),
            "gt_positive_total": int(gt_pos), "pred_positive_total": int(pred_pos),
        },
        "per_frame": frame_metrics,
    }


def draw_bboxes_on_panel(pil_img, x_offset, bboxes, colors):
    draw = ImageDraw.Draw(pil_img)
    for (bx0, by0, bx1, by1, inst_idx) in bboxes:
        color = colors[inst_idx % len(colors)]
        px0 = x_offset + bx0 * sx
        py0 = HEADER_H + by0 * sy
        px1 = x_offset + bx1 * sx
        py1 = HEADER_H + by1 * sy
        draw.rectangle([px0, py0, px1, py1], outline=color, width=2)


def save_eval_data(json_path, sample, cond_channel, metrics):
    record = {
        "orig_channel": sample["channel"],
        "video_name": sample["video_name"],
        "file_id": sample["file_id"],
        "cond_channel": cond_channel,
        "n_instances": len(sample["instances"]),
        "duration": sample["duration"],
        "n_frames": len(metrics["per_frame"]),
        "model_ckpt": CKPT_PATH,
        "aggregate": metrics["aggregate"],
        "per_frame": metrics["per_frame"],
    }
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)


print("✅ 헬퍼 함수 정의 완료")


In [ ]:
# %% 셀 3: 랜덤 1 영상 — 5패널 (Pred BBox는 CNN params 직접 사용)
candidates = [s for s in eval_samples if len(s["instances"]) > 0]
sample = random.choice(candidates)
orig_ch = sample["channel"]
file_id = sample["file_id"]
instances = sample["instances"]
n_inst = len(instances)

print(f"🎲 선택: [{orig_ch}] {file_id}")
print(f"   인스턴스: {n_inst}개  duration: {sample['duration']:.1f}s")

inputs = prepare_inputs(sample)
n_frames = inputs["n_frames"]
telop_mask = inputs["telop_mask"]
active_matrix = inputs["active_matrix"]
slot_inst_map = inputs["slot_inst_map"]
gt_masks = inputs["gt_masks"]

inst_colors = get_instance_colors(n_inst)
orig_ch_id = channel2id[orig_ch]

result = run_inference(inputs, orig_ch_id)
metrics = compute_eval_metrics(gt_masks, result["pred_merged_bbox"], active_matrix)
agg = metrics["aggregate"]
print(f"   CNN bbox 기반: P={agg['P']:.3f}  R={agg['R']:.3f}  F1={agg['F1']:.3f}  IoU={agg['IoU']:.3f}")

# heatmap 기반 비교
metrics_hm = compute_eval_metrics(gt_masks, result["merged_heatmap"], active_matrix)
agg_hm = metrics_hm["aggregate"]
print(f"   Heatmap thresh 기반: P={agg_hm['P']:.3f}  R={agg_hm['R']:.3f}  F1={agg_hm['F1']:.3f}  IoU={agg_hm['IoU']:.3f}")

inst_bboxes = []
for j in range(n_inst):
    cx, cy = int(instances[j]["x"]), int(instances[j]["y"])
    w, h = int(instances[j]["w"]), int(instances[j]["h"])
    x0 = max(0, cx - w // 2)
    y0 = max(0, cy - h // 2)
    x1 = min(GRID_W, x0 + w)
    y1 = min(GRID_H, y0 + h)
    inst_bboxes.append((x0, y0, x1, y1))

frame_dir = os.path.join(FRAME_DIR, orig_ch, file_id)
out_path = f"{OUTPUT_DIR}/_random_5panel.mp4"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(out_path, fourcc, VIS_FPS, (total_w, total_h))

px = [i * (VIS_W + GAP) for i in range(N_PANELS)]

for fi in range(n_frames):
    canvas = np.zeros((total_h, total_w, 3), dtype=np.uint8)

    frame_path = os.path.join(frame_dir, f"frame_{fi+1:08d}.jpg")
    if os.path.exists(frame_path):
        orig = cv2.resize(cv2.imread(frame_path), (VIS_W, VIS_H))
    else:
        orig = np.zeros((VIS_H, VIS_W, 3), dtype=np.uint8)
    canvas[HEADER_H:, px[0]:px[0]+VIS_W] = orig

    mh = result["merged_heatmap"][fi]
    mh_resized = cv2.resize(mh, (VIS_W, VIS_H), interpolation=cv2.INTER_LINEAR)
    mh_panel = (cmap_merged(mh_resized)[:, :, :3] * 255).astype(np.uint8)
    canvas[HEADER_H:, px[2]:px[2]+VIS_W] = cv2.cvtColor(mh_panel, cv2.COLOR_RGB2BGR)

    slot_panel = np.zeros((VIS_H, VIS_W, 3), dtype=np.float32)
    active_slots = np.where(telop_mask[fi])[0]
    for si in active_slots:
        inst_idx = slot_inst_map[fi, si]
        if inst_idx < 0:
            continue
        slot_2d = result["slot_y"][fi, si, :, None] * result["slot_x"][fi, si, None, :]
        slot_resized = cv2.resize(slot_2d, (VIS_W, VIS_H), interpolation=cv2.INTER_LINEAR)
        color = np.array(inst_colors[inst_idx % len(inst_colors)], dtype=np.float32) / 255.0
        slot_panel += slot_resized[:, :, None] * color[None, None, :]
    slot_panel = np.clip(slot_panel, 0, 1)
    canvas[HEADER_H:, px[3]:px[3]+VIS_W] = cv2.cvtColor(
        (slot_panel * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

    pil_img = Image.fromarray(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil_img)

    for i, name in enumerate(PANEL_NAMES):
        draw.text((px[i] + 6, 5), name, font=PIL_FONT, fill=(255, 255, 255))

    t_sec = fi / FPS
    n_active = int(active_matrix[fi].sum()) if active_matrix.shape[1] > 0 else 0
    draw.text((6, 28),
              f"t={t_sec:.1f}s  active={n_active}  [{orig_ch}]  "
              f"CNN-F1={agg['F1']:.3f}  HM-F1={agg_hm['F1']:.3f}",
              font=PIL_FONT_SM, fill=(180, 180, 180))

    if active_matrix.shape[1] > 0:
        active_inst = np.where(active_matrix[fi])[0]
        gt_boxes = [(inst_bboxes[j][0], inst_bboxes[j][1],
                     inst_bboxes[j][2], inst_bboxes[j][3], j) for j in active_inst]
        draw_bboxes_on_panel(pil_img, px[1], gt_boxes, inst_colors)

    # Pred BBox panel — CNN params 직접 사용 (threshold-free)
    for si in active_slots:
        inst_idx = slot_inst_map[fi, si]
        if inst_idx < 0:
            continue
        cx, cy, w, h = result["params"][fi, si]
        bx0 = int(np.clip(round(cx - w/2), 0, GRID_W))
        by0 = int(np.clip(round(cy - h/2), 0, GRID_H))
        bx1 = int(np.clip(round(cx + w/2), 0, GRID_W))
        by1 = int(np.clip(round(cy + h/2), 0, GRID_H))
        if bx1 > bx0 and by1 > by0:
            draw_bboxes_on_panel(pil_img, px[4],
                                 [(bx0, by0, bx1, by1, inst_idx)], inst_colors)

    canvas = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    writer.write(canvas)

writer.release()
print(f"✅ 저장: {out_path} ({n_frames} frames)")


In [ ]:
# %% 셀: 전체 eval set 추론 + JSON (CNN bbox 기반 메인 메트릭)
for si, sample in enumerate(tqdm(eval_samples, desc="전체 eval")):
    orig_ch = sample["channel"]
    file_id = sample["file_id"]
    orig_ch_id = channel2id[orig_ch]

    swap_ch = get_swap_channel(orig_ch, sample["video_name"], channels)
    swap_ch_id = channel2id[swap_ch]

    inputs = prepare_inputs(sample)

    orig_json = os.path.join(OUTPUT_DIR, orig_ch, file_id, f"{orig_ch}.json")
    if not os.path.exists(orig_json):
        result = run_inference(inputs, orig_ch_id)
        metrics_orig = compute_eval_metrics(
            inputs["gt_masks"], result["pred_merged_bbox"], inputs["active_matrix"])
        save_eval_data(orig_json, sample, orig_ch, metrics_orig)

    swap_json = os.path.join(OUTPUT_DIR, orig_ch, file_id, f"{swap_ch}.json")
    if not os.path.exists(swap_json):
        result = run_inference(inputs, swap_ch_id)
        metrics_swap = compute_eval_metrics(
            inputs["gt_masks"], result["pred_merged_bbox"], inputs["active_matrix"])
        save_eval_data(swap_json, sample, swap_ch, metrics_swap)

print(f"\n✅ 완료: {len(eval_samples)}개 영상 × 2 (orig+swap) → {OUTPUT_DIR}/")


In [ ]:
# %% 셀 4: JSON 로드 + 채널 density bucket
import os, json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

orig_map = {}
swap_map = {}
json_count = 0

for channel in tqdm(sorted(os.listdir(OUTPUT_DIR)), desc="JSON 로드"):
    ch_dir = os.path.join(OUTPUT_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for file_id in os.listdir(ch_dir):
        vid_dir = os.path.join(ch_dir, file_id)
        if not os.path.isdir(vid_dir):
            continue
        for fname in os.listdir(vid_dir):
            if not fname.endswith(".json"):
                continue
            path = os.path.join(vid_dir, fname)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    rec = json.load(f)
            except:
                continue
            key = (rec["orig_channel"], rec.get("file_id", rec.get("video_name", "")))
            if rec["orig_channel"] == rec["cond_channel"]:
                orig_map[key] = rec
            else:
                swap_map[key] = rec
            json_count += 1

print(f"\n✅ JSON 로드: {json_count:,}개")
print(f"   orig: {len(orig_map):,}개  swap: {len(swap_map):,}개")

paired_keys = sorted(set(orig_map.keys()) & set(swap_map.keys()))
print(f"   페어: {len(paired_keys):,}개")

channel_stats = {}
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    n_videos = 0
    density_sum = 0.0
    for fname in os.listdir(ch_dir):
        if not fname.endswith(".json"):
            continue
        try:
            with open(os.path.join(ch_dir, fname), "r") as f:
                data = json.load(f)
        except:
            continue
        instances = data.get("instances", [])
        n_videos += 1
        if instances:
            starts = np.array([inst["start_sec"] for inst in instances])
            ends = np.array([inst["end_sec"] for inst in instances])
            durs = np.clip(ends - starts, 0, None)
            vid_len = float(ends.max())
            if vid_len > 0:
                density_sum += float(durs.sum()) / vid_len
    if n_videos > 0:
        channel_stats[channel] = {"n_videos": n_videos, "avg_density": density_sum / n_videos}

stats_df = pd.DataFrame([{"channel": ch, **s} for ch, s in channel_stats.items()])
stats_df = stats_df.sort_values("avg_density").reset_index(drop=True)
N_BUCKETS = 5
stats_df["bucket"] = pd.qcut(stats_df["avg_density"], q=N_BUCKETS, labels=list(range(N_BUCKETS))).astype(int)
channel2bucket = dict(zip(stats_df["channel"], stats_df["bucket"]))

print(f"\n📊 채널 density bucket 분포")
print(stats_df.groupby("bucket")["avg_density"].agg(["count", "min", "max", "mean"]).to_string())


In [ ]:
# %% 셀 5: 절대적 품질 — Pixel-level F1 + IoU (CNN bbox 기반)
results = []
for key in paired_keys:
    rec = orig_map[key]
    agg = rec["aggregate"]
    results.append({
        "channel": key[0], "file_id": key[1],
        "n_instances": rec["n_instances"],
        "P": agg["P"], "R": agg["R"], "F1": agg["F1"],
        "IoU": agg["IoU"], "coverage_ratio": agg["coverage_ratio"],
        "gt_positive": agg["gt_positive_total"],
        "pred_positive": agg["pred_positive_total"],
    })

print(f"📊 Pixel-level 메트릭 (GT vs pred_orig, CNN bbox 기반, n={len(results):,})")
print(f"\n  {'메트릭':<20} {'mean':>8} {'median':>8} {'std':>8} {'min':>8} {'max':>8}")
print(f"  {'─'*20} {'─'*8} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for col in ["P", "R", "F1", "IoU", "coverage_ratio"]:
    vals = np.array([r[col] for r in results])
    print(f"  {col:<20} {vals.mean():>8.3f} {np.median(vals):>8.3f} {vals.std():>8.3f} {vals.min():>8.3f} {vals.max():>8.3f}")

f1s = np.array([r["F1"] for r in results])
print(f"\n📊 F1 분포")
print(f"  {'구간':<15} {'샘플수':>8} {'비율':>8}")
print(f"  {'─'*15} {'─'*8} {'─'*8}")
for (lo, hi), label in zip(
    [(0,0.1),(0.1,0.2),(0.2,0.4),(0.4,0.6),(0.6,0.8),(0.8,0.9),(0.9,1.001)],
    ["0~0.1","0.1~0.2","0.2~0.4","0.4~0.6","0.6~0.8","0.8~0.9","0.9~1.0"]):
    mask = (f1s >= lo) & (f1s < hi)
    print(f"  {label:<15} {mask.sum():>8} {mask.sum()/len(f1s)*100:>7.1f}%")

gt_ns = np.array([r["n_instances"] for r in results])
ious = np.array([r["IoU"] for r in results])

print(f"\n📊 인스턴스 수 구간별")
print(f"  {'구간':<15} {'n':>6} {'F1 mean':>10} {'F1 med':>10} {'IoU mean':>10}")
print(f"  {'─'*15} {'─'*6} {'─'*10} {'─'*10} {'─'*10}")
for i, (lo, hi) in enumerate([(0,1),(1,10),(10,100),(100,1000),(1000,10000)]):
    mask = (gt_ns >= lo) & (gt_ns < hi)
    if mask.sum() == 0: continue
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {f1s[mask].mean():>10.3f} {np.median(f1s[mask]):>10.3f} {ious[mask].mean():>10.3f}")

gt_zero = gt_ns == 0
gt_nonzero = gt_ns > 0
print(f"\n📊 GT=0: {gt_zero.sum()}개")
pred_pos_zero = np.array([r["pred_positive"] for r in results])[gt_zero]
print(f"  pred=0: {(pred_pos_zero == 0).sum()} ({(pred_pos_zero == 0).sum()/max(gt_zero.sum(),1)*100:.1f}%)")
print(f"  pred>0: {(pred_pos_zero > 0).sum()} ({(pred_pos_zero > 0).sum()/max(gt_zero.sum(),1)*100:.1f}%) ← false positive")
print(f"\n📊 GT>0: {gt_nonzero.sum()}개")
print(f"  F1 mean: {f1s[gt_nonzero].mean():.3f}, median: {np.median(f1s[gt_nonzero]):.3f}")
print(f"  IoU mean: {ious[gt_nonzero].mean():.3f}, median: {np.median(ious[gt_nonzero]):.3f}")


In [ ]:
# %% 셀 6: 상대적 차이 — Wilcoxon (orig vs swap)
from scipy.stats import wilcoxon

rows_d = []
for key in paired_keys:
    orig_agg = orig_map[key]["aggregate"]
    swap_agg = swap_map[key]["aggregate"]
    bucket = channel2bucket.get(key[0], -1)
    rows_d.append({
        "channel": key[0], "file_id": key[1], "bucket": bucket,
        "n_instances": orig_map[key]["n_instances"],
        "orig_F1": orig_agg["F1"], "swap_F1": swap_agg["F1"],
        "orig_IoU": orig_agg["IoU"], "swap_IoU": swap_agg["IoU"],
        "orig_coverage": orig_agg["coverage_ratio"],
        "swap_coverage": swap_agg["coverage_ratio"],
    })

orig_f1  = np.array([r["orig_F1"] for r in rows_d])
swap_f1  = np.array([r["swap_F1"] for r in rows_d])
orig_iou = np.array([r["orig_IoU"] for r in rows_d])
swap_iou = np.array([r["swap_IoU"] for r in rows_d])
orig_cov = np.array([r["orig_coverage"] for r in rows_d])
swap_cov = np.array([r["swap_coverage"] for r in rows_d])

delta_f1  = orig_f1 - swap_f1
delta_iou = orig_iou - swap_iou
delta_cov = np.abs(1 - swap_cov) - np.abs(1 - orig_cov)

gt_ns_d   = np.array([r["n_instances"] for r in rows_d])
buckets_d = np.array([r["bucket"] for r in rows_d])

print(f"📊 상대적 차이 (Δ = orig - swap, n={len(rows_d):,})")
print(f"\n  {'메트릭':<20} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8} {'Wilcoxon p':>12}")
print(f"  {'─'*20} {'─'*10} {'─'*10} {'─'*8} {'─'*12}")
for name, delta in [("F1", delta_f1), ("IoU", delta_iou), ("Coverage (|1-x|)", delta_cov)]:
    pos_pct = (delta > 0).sum() / len(delta) * 100
    nonzero = delta[delta != 0]
    p_str = f"{wilcoxon(nonzero).pvalue:.2e}" if len(nonzero) > 10 else "N/A"
    print(f"  {name:<20} {delta.mean():>+10.4f} {np.median(delta):>+10.4f} {pos_pct:>7.1f}% {p_str:>12}")

print(f"\n📊 F1 상세")
print(f"  {'':>15} {'orig':>10} {'swap':>10}")
print(f"  {'mean':>15} {orig_f1.mean():>10.3f} {swap_f1.mean():>10.3f}")
print(f"  {'median':>15} {np.median(orig_f1):>10.3f} {np.median(swap_f1):>10.3f}")

print(f"\n📊 IoU 상세")
print(f"  {'':>15} {'orig':>10} {'swap':>10}")
print(f"  {'mean':>15} {orig_iou.mean():>10.3f} {swap_iou.mean():>10.3f}")
print(f"  {'median':>15} {np.median(orig_iou):>10.3f} {np.median(swap_iou):>10.3f}")

print(f"\n📊 Density bucket별 Δ F1")
print(f"  {'bucket':<10} {'n':>6} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8} {'Wilcoxon p':>12}")
print(f"  {'─'*10} {'─'*6} {'─'*10} {'─'*10} {'─'*8} {'─'*12}")
for b in range(N_BUCKETS):
    mask = buckets_d == b
    if mask.sum() == 0: continue
    d = delta_f1[mask]
    nonzero = d[d != 0]
    p_str = f"{wilcoxon(nonzero).pvalue:.2e}" if len(nonzero) > 10 else "N/A"
    print(f"  {b:<10} {mask.sum():>6} {d.mean():>+10.4f} {np.median(d):>+10.4f} {(d>0).sum()/len(d)*100:>7.1f}% {p_str:>12}")

print(f"\n📊 인스턴스 수 구간별 Δ F1")
print(f"  {'구간':<15} {'n':>6} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8}")
print(f"  {'─'*15} {'─'*6} {'─'*10} {'─'*10} {'─'*8}")
for i, (lo, hi) in enumerate([(0,1),(1,10),(10,100),(100,1000),(1000,10000)]):
    mask = (gt_ns_d >= lo) & (gt_ns_d < hi)
    if mask.sum() == 0: continue
    d = delta_f1[mask]
    print(f"  {lo:>5}~{hi:<8} {mask.sum():>6} {d.mean():>+10.4f} {np.median(d):>+10.4f} {(d>0).sum()/len(d)*100:>7.1f}%")

print(f"\n📊 Density bucket별 Δ IoU")
print(f"  {'bucket':<10} {'n':>6} {'Δ mean':>10} {'Δ median':>10} {'Δ>0 %':>8}")
print(f"  {'─'*10} {'─'*6} {'─'*10} {'─'*10} {'─'*8}")
for b in range(N_BUCKETS):
    mask = buckets_d == b
    if mask.sum() == 0: continue
    d = delta_iou[mask]
    print(f"  {b:<10} {mask.sum():>6} {d.mean():>+10.4f} {np.median(d):>+10.4f} {(d>0).sum()/len(d)*100:>7.1f}%")
